# 🚀 FinDoc Intelligence - GPU Acceleration Setup

This notebook sets up Ollama with GPU acceleration and exposes it via ngrok tunnel.

**Benefits:**
- ⚡ 10-20x faster inference (Llama: 2-3 sec vs 33 sec)
- 🎮 Free T4 GPU (or A100 with Colab Pro)
- 🔌 Easy integration with your local backend

**Before you start:**
1. Make sure GPU is enabled: Runtime → Change runtime type → T4 GPU
2. Get a free ngrok account: https://dashboard.ngrok.com/signup
3. Get your auth token: https://dashboard.ngrok.com/get-started/your-authtoken

**⚠️ IMPORTANT: ngrok Free Tier Limitation**
- ngrok free tier shows a browser warning page that blocks API requests
- You'll need to either:
  - Option A: Upgrade to ngrok paid plan ($8/month) - removes warning
  - Option B: Use Cloudflare Tunnel (free, no warning) - see alternative setup below
  - Option C: Use ngrok with custom header workaround (implemented in this notebook)

In [1]:
# ============================================================================
# Step 1: Install Dependencies and Ollama
# ============================================================================
print("📦 Installing zstd (required for Ollama)...")
!apt-get update -qq && apt-get install -y -qq zstd > /dev/null 2>&1
print("✅ zstd installed")

print("\n📦 Installing Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh
print("✅ Ollama installed")

📦 Installing zstd (required for Ollama)...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
✅ zstd installed

📦 Installing Ollama...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
✅ Ollama installed


In [2]:
# ============================================================================
# Step 2: Start Ollama Server
# ============================================================================
import subprocess
import time

print("🚀 Starting Ollama server...")

# Start Ollama serve in background
ollama_process = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait for server to start
time.sleep(5)
print("✅ Ollama server started on port 11434")

🚀 Starting Ollama server...
✅ Ollama server started on port 11434


In [3]:
# ============================================================================
# Step 3: Pull Models
# ============================================================================
print("📥 Downloading Llama 3.2:3b model...")
print("(This may take 2-3 minutes)")
!ollama pull llama3.2:3b

print("\n📥 Downloading Gemma 2:9b model...")
print("(This may take 5-7 minutes)")
!ollama pull gemma2:9b

print("\n✅ All models downloaded!")

📥 Downloading Llama 3.2:3b model...
(This may take 2-3 minutes)


📥 Downloading Gemma 2:9b model...
(This may take 5-7 minutes)


✅ All models downloaded!


In [4]:
# ============================================================================
# Step 4: Test Ollama Locally
# ============================================================================
print("🧪 Testing Ollama with Llama model...")
!ollama run llama3.2:3b "Say 'GPU test successful!' and nothing else" --verbose=false
print("\n✅ Ollama is working!")

🧪 Testing Ollama with Llama model...
GPU test successful!


✅ Ollama is working!


In [5]:
# ============================================================================
# Step 5: Check GPU
# ============================================================================
print("🎮 Checking GPU availability...")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
print("\n✅ GPU is available and ready!")

🎮 Checking GPU availability...
name, memory.total [MiB], memory.free [MiB]
Tesla T4, 15360 MiB, 12112 MiB

✅ GPU is available and ready!


In [14]:
# Alternative: Use Cloudflare Tunnel (no browser warning!)
print("🌐 Setting up Cloudflare Tunnel...\n")

# Install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

print("✅ Cloudflared installed\n")

# Start tunnel in background
import subprocess
import time
import re

print("🚀 Starting Cloudflare Tunnel...")
print("(This may take 10-15 seconds)\n")

# Start cloudflared tunnel
tunnel_process = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:11434'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True
)

# Wait for tunnel URL
tunnel_url = None
for i in range(30):
    line = tunnel_process.stdout.readline()
    if 'trycloudflare.com' in line:
        # Extract URL from output
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            break
    time.sleep(0.5)

if tunnel_url:
    print(f"✅ Cloudflare Tunnel created!\n")
    print("="*70)
    print("🎉 YOUR OLLAMA URL (NO BROWSER WARNING!):")
    print("="*70)
    print(f"\n{tunnel_url}\n")
    print("📝 Update your backend/.env file:")
    print(f"   OLLAMA_BASE_URL={tunnel_url}")
    print("\n⚠️  Keep this notebook running!")
    print("="*70)
else:
    print("❌ Failed to get tunnel URL. Check the output above for errors.")

🌐 Setting up Cloudflare Tunnel...

✅ Cloudflared installed

🚀 Starting Cloudflare Tunnel...
(This may take 10-15 seconds)

✅ Cloudflare Tunnel created!

🎉 YOUR OLLAMA URL (NO BROWSER WARNING!):

https://italia-reed-sole-missile.trycloudflare.com

📝 Update your backend/.env file:
   OLLAMA_BASE_URL=https://italia-reed-sole-missile.trycloudflare.com

⚠️  Keep this notebook running!


In [ ]:
# Keep Alive - Run this to keep Ollama and Cloudflare Tunnel active
import time

print("⏳ Keeping Ollama and Cloudflare Tunnel alive...")
print("💡 Keep this cell running while using the API")
print("🛑 Press the Stop button when you're done\n")
print("Status: ", end="", flush=True)

try:
    counter = 0
    while True:
        time.sleep(60)
        counter += 1
        print(f"✓ {counter}m", end=" ", flush=True)
        if counter % 10 == 0:
            print("\nStatus: ", end="", flush=True)
except KeyboardInterrupt:
    print("\n\n🛑 Stopped")
